# первый замер: готовый bi-encoder без обучения

берем предобученную двухбашенную модель как есть, кодируем корпус и запросы, ищем ближайшие по косинусу и считаем Recall@50. обучения тут нет. цель, получить точку отсчета и понять, с какой модели дообучать

заодно здесь собирается валидация, которой потом будут пользоваться все остальные блокноты

кодировать весь корпус займет часы, поэтому меряем на подмножестве: все позитивы валидации плюс 45к случайных объявлений бенчмарка. число получится оптимистичнее, чем на полном корпусе, но для сравнения моделей между собой и до/после обучения этого достаточно. полный корпус закодируем один раз финальной моделью


In [1]:
import sys, time
sys.path.insert(0, "..")

from pathlib import Path
import numpy as np
import polars as pl
import torch
from sentence_transformers import SentenceTransformer

from src.split import build_validation, QUERY_COLS
from src.text import item_text, query_text, normalize
from src.metrics import recall_at_k, recall_by_slice
from src.search import encode, topk

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(exist_ok=True)

# какую модель меряем. у e5 запрос и документ помечаются префиксами, у bge, нет
MODELS = {
    "user-bge-m3": dict(name="deepvk/USER-bge-m3", q="", p=""),
    "bge-m3": dict(name="BAAI/bge-m3", q="", p=""),
    "e5-large-instruct": dict(name="intfloat/multilingual-e5-large-instruct",
                              q="Instruct: Given a search query, retrieve service ads that match it\nQuery: ", p=""),
    "e5-base": dict(name="intfloat/multilingual-e5-base", q="query: ", p="passage: "),
}
MODEL_KEY = "user-bge-m3"
MAX_LEN = 160 # медиана объявления 299 токенов, режем сильно, зато втрое быстрее
pl.Config(fmt_str_lengths=90)

/Users/amirhamidullin/PycharmProjects/avito_recsys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
from .autonotebook import tqdm as notebook_tqdm


## валидация

2500 запросов, как в бенчмарке. 37% из них с текстом, который остается в трейне (seen), остальные с уникальным (unseen). Один раз собираем и сохраняем, дальше все блокноты читают готовое

In [2]:
val_path = PROC / "val_queries.parquet"
train_path = PROC / "train_rest.parquet"

if val_path.exists():
    val = pl.read_parquet(val_path)
    print("валидация уже собрана, читаю с диска")
else:
    train = pl.read_parquet(RAW / "train.parquet")
    train_rest, val = build_validation(train, n_val=2500, seen_share=0.37, seed=42)
    val.write_parquet(val_path)
    train_rest.write_parquet(train_path)
    print("трейн после изъятия валидации", len(train_rest), "строк")

print("валидационных запросов", len(val))
print("из них seen", val["is_seen"].sum(), "unseen", (~val["is_seen"]).sum())
print("релевантных на запрос в среднем", round(val["relevant"].list.len().mean(), 2))
val.head(5)

валидация уже собрана, читаю с диска
валидационных запросов 2500
из них seen 925  unseen 1575
релевантных на запрос в среднем 1.15


query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,relevant,is_seen
str,str,i64,i32,str,i64,list[str],bool
"""0""","""комната лофт""",625810,0,"""Вид услуги Праздники, мероприятия""",114,"[""fb1d389a75d4db52""]",false
"""1""","""наращивание ногтей верхняя терасса""",659540,0,"""Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье""",114,"[""36cd251a1a8815b4""]",false
"""2""","""установка навигатора чанган""",621540,0,"""""",114,"[""f6c54a3a0ee23dde""]",false
"""3""","""териберка выход в море""",653240,0,"""Вид услуги Праздники, мероприятия""",114,"[""1a6373ff41d320d6""]",false
"""4""","""гид сочи поход""",634450,0,"""""",114,"[""76ec3be18b8bceca""]",false


## корпус для валидации

все объявления бенчмарка плюс позитивы валидационных запросов. бенчмарковые объявления без меток, честные отвлекающие: их 189 тыс, и 90% из них модель на обучении не видела. Так валидация повторяет холодность настоящего корпуса

In [ ]:
ITEM_COLS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text","item_category_id", "item_microcat_id", "item_price", "item_rating","item_rating_reviews_count", "item_location_id", "item_latitude", "item_longitude", "item_is_phone_hidden", "item_is_message_forbidden"]

bench_items = pl.read_parquet(RAW / "benchmark_items.parquet").select(ITEM_COLS)

val_item_ids = set(val["relevant"].explode().to_list())
val_items = (pl.scan_parquet(RAW / "train.parquet").select(ITEM_COLS).filter(pl.col("item_id").is_in(list(val_item_ids))).unique(subset="item_id").collect())

# если объявление есть и там и там, берем бенчмарковую версию
corpus_path = PROC / "val_corpus.parquet"
if corpus_path.exists():
    corpus = pl.read_parquet(corpus_path)
else:
    corpus = pl.concat([bench_items, val_items.filter(~pl.col("item_id").is_in(bench_items["item_id"]))]).sort("item_id")
    corpus.write_parquet(corpus_path)

print("объявлений бенчмарка", len(bench_items))
print("позитивов валидации", len(val_items), "из них уже были в бенчмарке", len(val_item_ids & set(bench_items["item_id"])))
print("корпус для валидации", len(corpus))

/var/folders/lz/_v3mf0j971z3l63wg5q45y8c0000gn/T/ipykernel_5104/3536833119.py:8: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
val_item_ids = set(val["relevant"].explode().to_list())


/var/folders/lz/_v3mf0j971z3l63wg5q45y8c0000gn/T/ipykernel_5104/3536833119.py:16: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
corpus = pl.concat([bench_items, val_items.filter(~pl.col("item_id").is_in(bench_items["item_id"]))])


объявлений бенчмарка 189212
позитивов валидации 2864  из них уже были в бенчмарке 191
корпус для валидации 191885


## тексты

что именно видит модель. заголовок, вид и тип услуги, список услуг из прайс-листа, начало описания. Подробно про то, почему такой порядок,, в архитектуре, раздел 3.4

In [4]:
corpus_texts = [item_text(t, p, d) for t, p, d in zip(corpus["item_title_raw"], corpus["item_infm_params_text"], corpus["item_description_raw"])]
query_texts = [query_text(q, p) for q, p in zip(val["search_query"], val["search_infm_params_text"])]

print("три объявления как их видит модель\n")
for i in [0, 1, 2]:
    print(corpus_texts[i][:300])
    print()

print("три запроса\n")
for i in [0, 1, 2]:
    print(query_texts[i])

lens = [len(t) for t in corpus_texts]
print("\nдлина текста объявления в символах: медиана", int(np.median(lens)), "максимум", max(lens))
# подмножество корпуса для быстрых замеров: все позитивы валидации и 45 тыс случайных
sub_path = PROC / "val_corpus_sub_idx.npy"
if sub_path.exists():
    sub_idx = np.load(sub_path)
else:
    rng = np.random.default_rng(0)
    must = set(i for rel in val["relevant"] for i in rel)
    ids = corpus["item_id"].to_list()
    must_idx = [j for j, i in enumerate(ids) if i in must]
    others = [j for j, i in enumerate(ids) if i not in must]
    sub_idx = np.array(sorted(must_idx + list(rng.choice(others, size=45_000, replace=False))))
    np.save(sub_path, sub_idx)
sub_texts = [corpus_texts[j] for j in sub_idx]
sub_ids = [corpus["item_id"][int(j)] for j in sub_idx]
print("\nподмножество корпуса для замера", len(sub_idx))

три объявления как их видит модель:

ремонт/выкуп компьют. и ноутбуков с выездом на дом. вид услуги: компьютерная помощь. тип услуги: компьютеры. услуги: диагностика компьютера, замена термопасты, ремонт после залития, чистка системы охлаждения, установка базовых драйверов, восстановление удаленных файлов, установка антивируса, чистка 

обучение ребенка чтению. вид услуги: обучение, курсы. тип услуги: детское развитие, логопеды. услуги: обучение чтению, обучение счету, обучение письму. дорогие родители и маленькие книголюбы! я, детский психолог/нейропсихолог, рада пригласить вас и ваших детей на захватывающее и увлекательное обучен

афрокудри 5+. вид услуги: красота, здоровье. тип услуги: услуги парикмахера. услуги: укладка афрокудри по плечи, укладка афрокудри по лопатки, укладка афрокудри по пояс, укладка афрокудри ниже пояса, доп плата за время до 10:00 и после 20:00, густые волосы. внимание укладка !!!! 🥳 афрокудри отличный

три запроса:

комната лофт. вид услуги: праздники, меропр

## сколько это будет считаться

прежде чем кодировать 190 тыс объявлений, замеряем скорость на тысяче и прикидываем время

In [5]:
cfg = MODELS[MODEL_KEY]
model = SentenceTransformer(cfg["name"], device="mps")
model.max_seq_length = MAX_LEN
model.half() # половинная точность на MPS почти вдвое быстрее, на качестве не сказывается
print("модель", cfg["name"])
print("размерность", model.get_sentence_embedding_dimension())

t = time.time()
_ = encode(model, sub_texts[:1024], batch_size=64, prefix=cfg["p"])
per_sec = 1024 / (time.time() - t)
print("скорость", round(per_sec), "текстов в секунду")
print("подмножество займет примерно", round(len(sub_texts) / per_sec / 60), "минут")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8953.48it/s]

/var/folders/lz/_v3mf0j971z3l63wg5q45y8c0000gn/T/ipykernel_5104/498565064.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
print("размерность", model.get_sentence_embedding_dimension())


модель deepvk/USER-bge-m3
размерность 1024



Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...


Batches:  88%|████████▊ | 14/16 [00:16<00:02,  1.17s/it]


Batches:  94%|█████████▍| 15/16 [00:17<00:01,  1.06s/it]


Batches: 100%|██████████| 16/16 [00:18<00:00,  1.10it/s]


Batches: 100%|██████████| 16/16 [00:18<00:00,  1.14s/it]

скорость 56 текстов в секунду
подмножество займет примерно 14 минут


## кодируем подмножество

эмбеддинги кладем на диск: они нужны потом для сравнения с дообученной моделью


In [6]:
emb_dir = PROC / "emb" / MODEL_KEY
emb_dir.mkdir(parents=True, exist_ok=True)
corpus_emb_path = emb_dir / "val_corpus_sub.npy"

if corpus_emb_path.exists():
    corpus_emb = np.load(corpus_emb_path)
    print("эмбеддинги подмножества уже есть, читаю с диска", corpus_emb.shape)
else:
    t = time.time()
    corpus_emb = encode(model, sub_texts, batch_size=64, prefix=cfg["p"])
    np.save(corpus_emb_path, corpus_emb)
    print("закодировали подмножество за", round((time.time() - t) / 60, 1), "минут", corpus_emb.shape)

query_emb = encode(model, query_texts, batch_size=64, prefix=cfg["q"])
np.save(emb_dir / "val_queries.npy", query_emb)


Batches:   0%|          | 0/748 [00:00<?, ?it/s]


Batches:   0%|          | 1/748 [00:01<14:38,  1.18s/it]


Batches:   0%|          | 2/748 [00:02<14:36,  1.17s/it]


Batches:   0%|          | 3/748 [00:03<14:34,  1.17s/it]


Batches:   1%|          | 4/748 [00:04<14:34,  1.18s/it]

...


Batches:  85%|████████▌ | 34/40 [00:05<00:00, 10.84it/s]


Batches:  90%|█████████ | 36/40 [00:05<00:00, 11.40it/s]


Batches:  95%|█████████▌| 38/40 [00:05<00:00, 11.50it/s]


Batches: 100%|██████████| 40/40 [00:06<00:00,  9.75it/s]


Batches: 100%|██████████| 40/40 [00:06<00:00,  6.52it/s]

## ищем и считаем

точный поиск: скалярное произведение запросов на весь корпус, топ по каждому. индекс не нужен

In [7]:
idx = topk(query_emb, corpus_emb, k=200)

predicted = {qid: [sub_ids[j] for j in row] for qid, row in zip(val["query_id"], idx)}
relevant = {qid: set(rel) for qid, rel in zip(val["query_id"], val["relevant"])}
is_seen = dict(zip(val["query_id"], val["is_seen"]))
has_filter = dict(zip(val["query_id"], val["search_infm_params_text"].str.strip_chars() != ""))

print("модель", MODEL_KEY, "без обучения, корпус", len(sub_ids), "\n")
for k in [10, 50, 100, 200]:
    print("recall@" + str(k) + "", round(recall_at_k(predicted, relevant, k), 4))

print("\nrecall@50 по срезам")
for name, flags in [("текст запроса был в трейне", is_seen), ("в запросе есть фильтр", has_filter)]:
    for value, (r, n) in recall_by_slice(predicted, relevant, flags, 50).items():
        print("", name, "=", value, "", round(r, 4), "(запросов", n, ")")

модель user-bge-m3 без обучения, корпус 47864 

recall@10 0.2664
recall@50 0.4859
recall@100 0.6023
recall@200 0.7253

recall@50 по срезам
   текст запроса был в трейне = False  0.5153   (запросов 1575 )
   текст запроса был в трейне = True  0.4358   (запросов 925 )
   в запросе есть фильтр = False  0.5927   (запросов 1121 )
   в запросе есть фильтр = True  0.3991   (запросов 1379 )


## посмотрим на промахи

несколько запросов, где правильное объявление не попало в топ-50: что искали, что было выбрано и что модель поставила наверх

In [8]:
title_by_id = dict(zip(corpus["item_id"], corpus["item_title_raw"]))
misses = [qid for qid in relevant if not (set(predicted[qid][:50]) & relevant[qid])]
print("промахов в топ-50", len(misses), "из", len(relevant), "\n")

rng = np.random.default_rng(0)
for qid in rng.choice(misses, size=6, replace=False):
    row = val.filter(pl.col("query_id") == qid).row(0, named=True)
    print("запрос", row["search_query"], "| фильтр", row["search_infm_params_text"] or "нет")
    print("выбрано", [title_by_id[i] for i in relevant[qid]][:2])
    print("модель поставила первыми", [title_by_id[i] for i in predicted[qid][:3]])
    # на каком месте оказалось правильное, если в топ-200 вообще есть
    ranks = [predicted[qid].index(i) + 1 for i in relevant[qid] if i in predicted[qid]]
    print("место правильного в топ-200", ranks if ranks else "нет и там")
    print()

промахов в топ-50 1241 из 2500 

запрос аренда тонар 40 тонн  | фильтр нет
выбрано ['Перевозка тонарами']
модель поставила первыми ['Самосвал. Тонар', 'Перевозка сыпучих грузов. Тонары, самосвалы', 'Перевозка сыпучих материалов. Тонары, самосвалы']
место правильного в топ-200 нет и там

запрос ремонт болгарок  | фильтр Тип услуги Другое Вид услуги Ремонт и обслуживание техники
выбрано ['Ремонт Бензоинструмента электроинструмент']
модель поставила первыми ['Ремонт бытовой техники', 'Ремонт электроники и бытовой техники', 'Ремонт сместитилей']
место правильного в топ-200 [59]

запрос обследование дома перед покупкой  | фильтр нет
выбрано ['Технадзор дом квартира строительство отделка']
модель поставила первыми ['Осмотр и проверка дома перед покупкой строителем', 'Экспертиза дома перед покупкой в Электроуглях', 'Экспертиза дома перед покупкой в Зарайске']
место правильного в топ-200 [89]

запрос мини швейное производство  | фильтр Вид услуги Бытовые услуги Тип услуги Пошив и ремонт одежды